# Main collection

Collect n unique subjects after `r/power.R` has locked sample size
from the largest of the eight pilot cell SDs (framework Section 6).

Do **not** pool the pilot into the eight primary TOSTs unless the design
is later amended. After this notebook, render `r/analysis.qmd` on the
main `evaluation.csv` and `ledger.csv`.

Needs CUDA.

**Google Colab:** Runtime → Change runtime type → GPU (T4 or L4). Run the
bootstrap cell first. Put `sample_size.json` from the pilot (or set `N`
by hand) under the Drive artifacts folder before collecting.

In [1]:
# Colab / local bootstrap.
# Colab: Runtime → Change runtime type → GPU, then run this cell.
# Local: skip clone and Drive if the repo is already on disk.

from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/RobinGirardin/zepto.git"
STUDY_BRANCH = "empirical-test"
DRIVE_STUDY_DIR = Path("MyDrive") / "zepto-apertus-validity"


def in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        marker = candidate / "studies" / "apertus-validity" / "python"
        if marker.is_dir():
            return candidate
    return None


IN_COLAB = in_colab()
REPO_ROOT = find_repo_root(Path.cwd().resolve())

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    ARTIFACTS_ROOT = Path("/content/drive") / DRIVE_STUDY_DIR
    ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)
else:
    ARTIFACTS_ROOT = None

if REPO_ROOT is None:
    if not IN_COLAB:
        raise FileNotFoundError(
            "could not find studies/apertus-validity; open this notebook from the repo"
        )
    clone_dir = Path("/content/zepto")
    if not (clone_dir / "studies" / "apertus-validity" / "python").is_dir():
        subprocess.check_call(
            [
                "git",
                "clone",
                "--branch",
                STUDY_BRANCH,
                "--depth",
                "1",
                REPO_URL,
                str(clone_dir),
            ]
        )
    REPO_ROOT = clone_dir

STUDY_ROOT = REPO_ROOT / "studies" / "apertus-validity"
if ARTIFACTS_ROOT is None:
    ARTIFACTS_ROOT = STUDY_ROOT / "artifacts"

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(STUDY_ROOT / "python"))

if IN_COLAB:
    # Do not pip-install torch: Colab already has a CUDA build.
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)]
    )
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", "transformers"]
    )

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "this notebook needs a CUDA GPU. In Colab: Runtime → Change runtime type → GPU."
    )

print("colab:", IN_COLAB)
print("repo:", REPO_ROOT)
print("study:", STUDY_ROOT)
print("artifacts:", ARTIFACTS_ROOT)
print("gpu:", torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

Mounted at /content/drive
colab: True
repo: /content/zepto
study: /content/zepto/studies/apertus-validity
artifacts: /content/drive/MyDrive/zepto-apertus-validity
gpu: Tesla T4 (7, 5)


In [5]:
import json

from apertus_validity import Catalog, run_collection

size_path = ARTIFACTS_ROOT / "pilot" / "sample_size.json"
if size_path.exists():
    N = int(json.loads(size_path.read_text())["n_subjects"])
    print("n from power.R:", N)
else:
    N = None
    print("sample_size.json not found; set N by hand")

SEED = 2
N = 18
OUT = ARTIFACTS_ROOT / "main"
catalog = Catalog()
N, OUT

sample_size.json not found; set N by hand


(18, PosixPath('/content/drive/MyDrive/zepto-apertus-validity/main'))

In [6]:
if N is None:
    raise ValueError("set N from power.R before collecting the main sample")

run_collection(N, seed=SEED, output_dir=OUT, catalog=catalog)
print("wrote", OUT)
print("next: quarto render studies/apertus-validity/r/analysis.qmd \\")
print("  -P evaluation:../artifacts/main/evaluation.csv \\")
print("  -P ledger:../artifacts/main/ledger.csv")

[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=32
[transformers] CUDA-fused xIELU not available (No module named 'xielu') – falling back to a Python version.
For CUDA xIELU (experimental), `pip install git+https://github.com/nickjbrowning/XIELU`
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=32
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=32
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=32
[transformers] `rope_parameters`'s original_max_position_embeddings field must be less than max_position_embeddings, got 8192 and max_position_embeddings=64
[transformers] `rop

wrote /content/drive/MyDrive/zepto-apertus-validity/main
next: quarto render studies/apertus-validity/r/analysis.qmd \
  -P evaluation:../artifacts/main/evaluation.csv \
  -P ledger:../artifacts/main/ledger.csv
